## Actividad 2: Balanceo de Carga y Gestión de Solicitudes

---

### 📋 Información General
- **Módulo:** Sistemas Distribuidos para IA
- **Duración:** 40 minutos
- **Objetivo:** Implementar un balanceador de carga básico para distribuir solicitudes entre nodos simulados
- **Alumno:** Juan David Valencia Martínez
- **Fecha:** 2026

---

## 🎯 Objetivos de Aprendizaje

Al completar esta actividad, serás capaz de:
1. Entender el concepto de **balanceo de carga** en sistemas distribuidos
2. Implementar un **balanceador de carga** básico usando Python
3. Simular **nodos de procesamiento** para manejar solicitudes concurrentes
4. Utilizar **aiohttp** para comunicación asíncrona entre nodos
5. Analizar la **distribución de solicitudes** y su impacto en el rendimiento

---

## 📚 Conceptos Clave

### ¿Qué es un Balanceador de Carga?
Un **balanceador de carga** es un componente que distribuye solicitudes de clientes entre múltiples servidores (nodos) para:
- **Optimizar recursos:** Distribuir la carga de trabajo uniformemente
- **Mejorar rendimiento:** Reducir tiempo de respuesta
- **Aumentar disponibilidad:** Si un nodo falla, otros pueden continuar
- **Escalar horizontalmente:** Manejar más solicitudes agregando más nodos

### Estrategias Comunes de Balanceo
1. **Round Robin:** Distribuye solicitudes de forma circular (1→2→3→1...)
2. **Random:** Selecciona un nodo aleatoriamente
3. **Least Connections:** Envía al nodo con menos conexiones activas
4. **IP Hash:** Usa la IP del cliente para consistencia
5. **Weighted Round Robin:** Asigna pesos según capacidad del nodo

En esta actividad implementaremos **Random** y **Round Robin**.


## 🔧 Instalación de Dependencias

Primero, instalamos las librerías necesarias para esta actividad.

In [1]:
# Instalar dependencias necesarias
# Ejecutar una sola vez

import subprocess
import sys

paquetes = ['aiohttp', 'asyncio', 'nest_asyncio', 'numpy']

for paquete in paquetes:
    print(f"Instalando {paquete}...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', paquete, '-q'])

print("✅ Todas las dependencias han sido instaladas correctamente.")

Instalando aiohttp...
Instalando asyncio...
Instalando nest_asyncio...
Instalando numpy...
✅ Todas las dependencias han sido instaladas correctamente.


## 📦 Importar Librerías

Ahora importamos todas las librerías necesarias para el desarrollo.

In [2]:
# Importar librerías estándar de Python
import asyncio
import time
import random
from typing import List, Dict, Tuple
from collections import defaultdict
from datetime import datetime

# Importar librerías externas
import numpy as np
import aiohttp
from aiohttp import web

# Permitir loops anidados en Jupyter
import nest_asyncio
nest_asyncio.apply()

print("✅ Todas las librerías han sido importadas correctamente.")

✅ Todas las librerías han sido importadas correctamente.


---

## 🏗️ Parte 1: Simular Nodos de Procesamiento

Primero, simulamos servidores que procesan solicitudes. Cada nodo recibe datos y realiza una predicción simulada.

In [3]:
# ============================================================================
# CLASE: SimulatedNode
# Descripción: Simula un nodo de procesamiento que recibe solicitudes
# ============================================================================

class SimulatedNode:
    """
    Representa un nodo de procesamiento simulado que:
    - Recibe solicitudes de inferencia
    - Procesa datos durante un tiempo simulado
    - Retorna predicciones
    - Mantiene estadísticas de uso
    """
    
    def __init__(self, node_id: int, port: int, processing_time: float = 0.5):
        """
        Inicializa un nodo de procesamiento.
        
        Args:
            node_id (int): Identificador único del nodo
            port (int): Puerto en el que corre el nodo
            processing_time (float): Tiempo simulado de procesamiento en segundos
        """
        self.node_id = node_id
        self.port = port
        self.processing_time = processing_time
        
        # Estadísticas del nodo
        self.total_requests = 0  # Total de solicitudes procesadas
        self.total_processing_time = 0  # Tiempo total de procesamiento
        self.active_connections = 0  # Conexiones activas en este momento
        self.request_log = []  # Log de solicitudes procesadas
    
    async def process_inference(self, request_data: dict) -> dict:
        """
        Procesa una solicitud de inferencia.
        
        Args:
            request_data (dict): Datos de la solicitud (incluye imagen simulada)
            
        Returns:
            dict: Resultado de la inferencia con metadatos
        """
        # Incrementar conexiones activas
        self.active_connections += 1
        
        # Registrar inicio del procesamiento
        start_time = time.time()
        request_id = request_data.get('request_id', 'unknown')
        
        # Simular procesamiento con delay
        await asyncio.sleep(self.processing_time)
        
        # Generar predicción simulada (números aleatorios entre 0 y 1)
        prediction = np.random.rand(1000).tolist()  # 1000 clases simuladas
        
        # Calcular tiempo de procesamiento
        end_time = time.time()
        processing_duration = end_time - start_time
        
        # Actualizar estadísticas
        self.total_requests += 1
        self.total_processing_time += processing_duration
        self.active_connections -= 1
        
        # Crear entrada de log
        log_entry = {
            'request_id': request_id,
            'timestamp': datetime.now().isoformat(),
            'processing_time': processing_duration,
            'top_prediction': float(np.max(prediction)),
            'predicted_class': int(np.argmax(prediction))
        }
        self.request_log.append(log_entry)
        
        # Retornar resultado
        return {
            'node_id': self.node_id,
            'request_id': request_id,
            'predictions': prediction[:10],  # Retornar solo top 10 para brevedad
            'predicted_class': int(np.argmax(prediction)),
            'confidence': float(np.max(prediction)),
            'processing_time': processing_duration
        }
    
    def get_statistics(self) -> dict:
        """
        Retorna estadísticas del nodo.
        
        Returns:
            dict: Estadísticas de uso del nodo
        """
        avg_processing_time = (
            self.total_processing_time / self.total_requests 
            if self.total_requests > 0 else 0
        )
        
        return {
            'node_id': self.node_id,
            'port': self.port,
            'total_requests': self.total_requests,
            'active_connections': self.active_connections,
            'avg_processing_time': avg_processing_time,
            'total_processing_time': self.total_processing_time
        }


# Crear instancias de nodos simulados
nodes = [
    SimulatedNode(node_id=1, port=8081, processing_time=0.3),
    SimulatedNode(node_id=2, port=8082, processing_time=0.5),
    SimulatedNode(node_id=3, port=8083, processing_time=0.4)
]

print("✅ Nodos de procesamiento creados:")
for node in nodes:
    print(f"   • Nodo {node.node_id} - Puerto: {node.port} - Tiempo procesamiento: {node.processing_time}s")

✅ Nodos de procesamiento creados:
   • Nodo 1 - Puerto: 8081 - Tiempo procesamiento: 0.3s
   • Nodo 2 - Puerto: 8082 - Tiempo procesamiento: 0.5s
   • Nodo 3 - Puerto: 8083 - Tiempo procesamiento: 0.4s


---

## 🎯 Parte 2: Implementar Balanceadores de Carga

Ahora implementamos dos estrategias de balanceo: **Random** y **Round Robin**.

In [4]:
# ============================================================================
# CLASE: LoadBalancer
# Descripción: Implementa diferentes estrategias de balanceo de carga
# ============================================================================

class LoadBalancer:
    """
    Balanceador de carga que distribuye solicitudes entre múltiples nodos
    usando diferentes estrategias.
    """
    
    def __init__(self, nodes: List[SimulatedNode], strategy: str = 'random'):
        """
        Inicializa el balanceador de carga.
        
        Args:
            nodes (List[SimulatedNode]): Lista de nodos disponibles
            strategy (str): Estrategia de balanceo ('random' o 'round_robin')
        """
        self.nodes = nodes
        self.strategy = strategy
        self.current_index = 0  # Índice para round robin
        
        # Estadísticas del balanceador
        self.total_requests = 0
        self.request_distribution = defaultdict(int)  # Solicitudes por nodo
        self.response_times = []  # Tiempos de respuesta
        self.failed_requests = 0  # Solicitudes fallidas
    
    def select_node(self) -> SimulatedNode:
        """
        Selecciona un nodo según la estrategia configurada.
        
        Returns:
            SimulatedNode: Nodo seleccionado
        """
        if self.strategy == 'random':
            # Estrategia RANDOM: seleccionar nodo al azar
            return random.choice(self.nodes)
        
        elif self.strategy == 'round_robin':
            # Estrategia ROUND ROBIN: distribuir de forma circular
            node = self.nodes[self.current_index]
            self.current_index = (self.current_index + 1) % len(self.nodes)
            return node
        
        elif self.strategy == 'least_connections':
            # Estrategia LEAST CONNECTIONS: enviar al nodo con menos conexiones
            return min(self.nodes, key=lambda n: n.active_connections)
        
        else:
            # Por defecto, usar random
            return random.choice(self.nodes)
    
    async def forward_request(self, request_data: dict) -> dict:
        """
        Recibe una solicitud del cliente y la reenvía al nodo seleccionado.
        
        Args:
            request_data (dict): Datos de la solicitud del cliente
            
        Returns:
            dict: Respuesta del nodo + metadatos de balanceo
        """
        # Registrar tiempo de inicio
        start_time = time.time()
        self.total_requests += 1
        
        try:
            # Seleccionar nodo según estrategia
            selected_node = self.select_node()
            self.request_distribution[selected_node.node_id] += 1
            
            # Procesar solicitud en el nodo seleccionado
            result = await selected_node.process_inference(request_data)
            
            # Calcular tiempo de respuesta total
            response_time = time.time() - start_time
            self.response_times.append(response_time)
            
            # Agregar metadatos al resultado
            result['balancer_metadata'] = {
                'strategy': self.strategy,
                'selected_node': selected_node.node_id,
                'total_response_time': response_time
            }
            
            return result
        
        except Exception as e:
            # Manejar errores
            self.failed_requests += 1
            print(f"❌ Error procesando solicitud: {str(e)}")
            return {'error': str(e)}
    
    def get_statistics(self) -> dict:
        """
        Retorna estadísticas del balanceador de carga.
        
        Returns:
            dict: Estadísticas detalladas
        """
        avg_response_time = (
            sum(self.response_times) / len(self.response_times)
            if self.response_times else 0
        )
        
        return {
            'strategy': self.strategy,
            'total_requests': self.total_requests,
            'failed_requests': self.failed_requests,
            'avg_response_time': avg_response_time,
            'request_distribution': dict(self.request_distribution),
            'nodes_statistics': [node.get_statistics() for node in self.nodes]
        }


print("✅ Clases de balanceador de carga implementadas correctamente.")

✅ Clases de balanceador de carga implementadas correctamente.


---

## 🧪 Parte 3: Pruebas de Balanceador - Estrategia RANDOM

Ahora ejecutamos pruebas con la estrategia **RANDOM**.

In [5]:
# ============================================================================
# Prueba 1: Balanceador con estrategia RANDOM
# ============================================================================

async def test_random_balancer():
    """
    Prueba el balanceador de carga con estrategia RANDOM.
    
    - Crea 10 solicitudes simuladas
    - Las distribuye aleatoriamente entre 3 nodos
    - Recopila y analiza las estadísticas
    """
    
    print("\n" + "="*80)
    print("PRUEBA 1: Balanceador de Carga - Estrategia RANDOM")
    print("="*80 + "\n")
    
    # Crear balanceador con estrategia random
    balancer = LoadBalancer(nodes=nodes, strategy='random')
    
    # Generar 10 solicitudes simuladas
    num_requests = 10
    print(f"📨 Enviando {num_requests} solicitudes con estrategia RANDOM...\n")
    
    # Ejecutar solicitudes de forma concurrente
    tasks = []
    for i in range(num_requests):
        # Crear datos de solicitud
        request_data = {
            'request_id': f'req_{i+1:03d}',
            'image': np.random.rand(224, 224, 3),  # Imagen simulada
            'timestamp': datetime.now().isoformat()
        }
        
        # Crear tarea asíncrona
        task = balancer.forward_request(request_data)
        tasks.append(task)
    
    # Esperar a que se completen todas las solicitudes
    results = await asyncio.gather(*tasks)
    
    # Mostrar resultados de algunas solicitudes
    print("\n📊 Primeras 3 solicitudes procesadas:")
    for i, result in enumerate(results[:3]):
        if 'error' not in result:
            print(f"\n   Solicitud {i+1}:")
            print(f"   - ID: {result['request_id']}")
            print(f"   - Nodo seleccionado: {result['balancer_metadata']['selected_node']}")
            print(f"   - Tiempo respuesta total: {result['balancer_metadata']['total_response_time']:.3f}s")
            print(f"   - Clase predicha: {result['predicted_class']}")
            print(f"   - Confianza: {result['confidence']:.4f}")
    
    # Obtener y mostrar estadísticas del balanceador
    stats = balancer.get_statistics()
    
    print(f"\n\n📈 ESTADÍSTICAS DEL BALANCEADOR:")
    print(f"\n   Estrategia: {stats['strategy'].upper()}")
    print(f"   Total solicitudes: {stats['total_requests']}")
    print(f"   Solicitudes fallidas: {stats['failed_requests']}")
    print(f"   Tiempo promedio respuesta: {stats['avg_response_time']:.4f}s")
    
    print(f"\n   Distribución de solicitudes por nodo:")
    for node_id, count in sorted(stats['request_distribution'].items()):
        percentage = (count / stats['total_requests']) * 100
        print(f"   - Nodo {node_id}: {count} solicitudes ({percentage:.1f}%)")
    
    print(f"\n   Estadísticas por nodo:")
    for node_stat in stats['nodes_statistics']:
        print(f"\n   Nodo {node_stat['node_id']} (Puerto {node_stat['port']}):")
        print(f"   - Solicitudes procesadas: {node_stat['total_requests']}")
        print(f"   - Tiempo promedio: {node_stat['avg_processing_time']:.4f}s")
        print(f"   - Conexiones activas: {node_stat['active_connections']}")
    
    return stats

# Ejecutar la prueba
stats_random = await test_random_balancer()


PRUEBA 1: Balanceador de Carga - Estrategia RANDOM

📨 Enviando 10 solicitudes con estrategia RANDOM...


📊 Primeras 3 solicitudes procesadas:

   Solicitud 1:
   - ID: req_001
   - Nodo seleccionado: 3
   - Tiempo respuesta total: 0.407s
   - Clase predicha: 107
   - Confianza: 0.9983

   Solicitud 2:
   - ID: req_002
   - Nodo seleccionado: 1
   - Tiempo respuesta total: 0.321s
   - Clase predicha: 579
   - Confianza: 0.9975

   Solicitud 3:
   - ID: req_003
   - Nodo seleccionado: 2
   - Tiempo respuesta total: 0.502s
   - Clase predicha: 983
   - Confianza: 0.9995


📈 ESTADÍSTICAS DEL BALANCEADOR:

   Estrategia: RANDOM
   Total solicitudes: 10
   Solicitudes fallidas: 0
   Tiempo promedio respuesta: 0.4183s

   Distribución de solicitudes por nodo:
   - Nodo 1: 2 solicitudes (20.0%)
   - Nodo 2: 3 solicitudes (30.0%)
   - Nodo 3: 5 solicitudes (50.0%)

   Estadísticas por nodo:

   Nodo 1 (Puerto 8081):
   - Solicitudes procesadas: 2
   - Tiempo promedio: 0.3199s
   - Conexiones a

---

## 🎯 Parte 4: Pruebas de Balanceador - Estrategia ROUND ROBIN

Ahora ejecutamos pruebas con la estrategia **ROUND ROBIN**.

In [6]:
# ============================================================================
# Prueba 2: Balanceador con estrategia ROUND ROBIN
# ============================================================================

# Reinicializar nodos para nueva prueba
nodes_rr = [
    SimulatedNode(node_id=1, port=8081, processing_time=0.3),
    SimulatedNode(node_id=2, port=8082, processing_time=0.5),
    SimulatedNode(node_id=3, port=8083, processing_time=0.4)
]

async def test_round_robin_balancer():
    """
    Prueba el balanceador de carga con estrategia ROUND ROBIN.
    
    - Crea 10 solicitudes simuladas
    - Las distribuye de forma circular entre 3 nodos
    - Recopila y analiza las estadísticas
    """
    
    print("\n" + "="*80)
    print("PRUEBA 2: Balanceador de Carga - Estrategia ROUND ROBIN")
    print("="*80 + "\n")
    
    # Crear balanceador con estrategia round robin
    balancer = LoadBalancer(nodes=nodes_rr, strategy='round_robin')
    
    # Generar 10 solicitudes simuladas
    num_requests = 10
    print(f"📨 Enviando {num_requests} solicitudes con estrategia ROUND ROBIN...\n")
    
    # Ejecutar solicitudes de forma concurrente
    tasks = []
    for i in range(num_requests):
        # Crear datos de solicitud
        request_data = {
            'request_id': f'req_{i+1:03d}',
            'image': np.random.rand(224, 224, 3),  # Imagen simulada
            'timestamp': datetime.now().isoformat()
        }
        
        # Crear tarea asíncrona
        task = balancer.forward_request(request_data)
        tasks.append(task)
    
    # Esperar a que se completen todas las solicitudes
    results = await asyncio.gather(*tasks)
    
    # Mostrar resultados de algunas solicitudes
    print("\n📊 Primeras 3 solicitudes procesadas:")
    for i, result in enumerate(results[:3]):
        if 'error' not in result:
            print(f"\n   Solicitud {i+1}:")
            print(f"   - ID: {result['request_id']}")
            print(f"   - Nodo seleccionado: {result['balancer_metadata']['selected_node']}")
            print(f"   - Tiempo respuesta total: {result['balancer_metadata']['total_response_time']:.3f}s")
            print(f"   - Clase predicha: {result['predicted_class']}")
            print(f"   - Confianza: {result['confidence']:.4f}")
    
    # Obtener y mostrar estadísticas del balanceador
    stats = balancer.get_statistics()
    
    print(f"\n\n📈 ESTADÍSTICAS DEL BALANCEADOR:")
    print(f"\n   Estrategia: {stats['strategy'].upper()}")
    print(f"   Total solicitudes: {stats['total_requests']}")
    print(f"   Solicitudes fallidas: {stats['failed_requests']}")
    print(f"   Tiempo promedio respuesta: {stats['avg_response_time']:.4f}s")
    
    print(f"\n   Distribución de solicitudes por nodo:")
    for node_id, count in sorted(stats['request_distribution'].items()):
        percentage = (count / stats['total_requests']) * 100
        print(f"   - Nodo {node_id}: {count} solicitudes ({percentage:.1f}%)")
    
    print(f"\n   Estadísticas por nodo:")
    for node_stat in stats['nodes_statistics']:
        print(f"\n   Nodo {node_stat['node_id']} (Puerto {node_stat['port']}):")
        print(f"   - Solicitudes procesadas: {node_stat['total_requests']}")
        print(f"   - Tiempo promedio: {node_stat['avg_processing_time']:.4f}s")
        print(f"   - Conexiones activas: {node_stat['active_connections']}")
    
    return stats

# Ejecutar la prueba
stats_rr = await test_round_robin_balancer()


PRUEBA 2: Balanceador de Carga - Estrategia ROUND ROBIN

📨 Enviando 10 solicitudes con estrategia ROUND ROBIN...


📊 Primeras 3 solicitudes procesadas:

   Solicitud 1:
   - ID: req_001
   - Nodo seleccionado: 1
   - Tiempo respuesta total: 0.305s
   - Clase predicha: 587
   - Confianza: 0.9996

   Solicitud 2:
   - ID: req_002
   - Nodo seleccionado: 2
   - Tiempo respuesta total: 0.502s
   - Clase predicha: 282
   - Confianza: 0.9991

   Solicitud 3:
   - ID: req_003
   - Nodo seleccionado: 3
   - Tiempo respuesta total: 0.402s
   - Clase predicha: 864
   - Confianza: 0.9996


📈 ESTADÍSTICAS DEL BALANCEADOR:

   Estrategia: ROUND_ROBIN
   Total solicitudes: 10
   Solicitudes fallidas: 0
   Tiempo promedio respuesta: 0.3934s

   Distribución de solicitudes por nodo:
   - Nodo 1: 4 solicitudes (40.0%)
   - Nodo 2: 3 solicitudes (30.0%)
   - Nodo 3: 3 solicitudes (30.0%)

   Estadísticas por nodo:

   Nodo 1 (Puerto 8081):
   - Solicitudes procesadas: 4
   - Tiempo promedio: 0.3050s
  

---

## 📊 Parte 5: Análisis Comparativo de Estrategias

Comparamos las dos estrategias implementadas.

In [7]:
# ============================================================================
# Análisis Comparativo entre estrategias
# ============================================================================

print("\n" + "="*80)
print("ANÁLISIS COMPARATIVO: RANDOM vs ROUND ROBIN")
print("="*80 + "\n")

# Comparar tiempos de respuesta promedio
print("📊 TIEMPO DE RESPUESTA PROMEDIO:")
print(f"   Random:       {stats_random['avg_response_time']:.4f}s")
print(f"   Round Robin:  {stats_rr['avg_response_time']:.4f}s")
diferencia = abs(stats_random['avg_response_time'] - stats_rr['avg_response_time'])
print(f"   Diferencia:   {diferencia:.4f}s\n")

# Comparar distribución de carga
print("📊 DISTRIBUCIÓN DE SOLICITUDES:")
print("\n   RANDOM:")
for node_id in sorted(stats_random['request_distribution'].keys()):
    count = stats_random['request_distribution'][node_id]
    print(f"   - Nodo {node_id}: {count} solicitudes")

print("\n   ROUND ROBIN:")
for node_id in sorted(stats_rr['request_distribution'].keys()):
    count = stats_rr['request_distribution'][node_id]
    print(f"   - Nodo {node_id}: {count} solicitudes")

# Análisis de utilización de nodos
print(f"\n\n📊 ANÁLISIS DE CARGA POR NODO:")

print("\n   RANDOM:")
for node_stat in stats_random['nodes_statistics']:
    print(f"\n   Nodo {node_stat['node_id']}:")
    print(f"   - Solicitudes: {node_stat['total_requests']}")
    print(f"   - Tiempo promedio: {node_stat['avg_processing_time']:.4f}s")

print("\n   ROUND ROBIN:")
for node_stat in stats_rr['nodes_statistics']:
    print(f"\n   Nodo {node_stat['node_id']}:")
    print(f"   - Solicitudes: {node_stat['total_requests']}")
    print(f"   - Tiempo promedio: {node_stat['avg_processing_time']:.4f}s")

print("\n" + "="*80)
print("FIN DEL ANÁLISIS COMPARATIVO")
print("="*80)


ANÁLISIS COMPARATIVO: RANDOM vs ROUND ROBIN

📊 TIEMPO DE RESPUESTA PROMEDIO:
   Random:       0.4183s
   Round Robin:  0.3934s
   Diferencia:   0.0249s

📊 DISTRIBUCIÓN DE SOLICITUDES:

   RANDOM:
   - Nodo 1: 2 solicitudes
   - Nodo 2: 3 solicitudes
   - Nodo 3: 5 solicitudes

   ROUND ROBIN:
   - Nodo 1: 4 solicitudes
   - Nodo 2: 3 solicitudes
   - Nodo 3: 3 solicitudes


📊 ANÁLISIS DE CARGA POR NODO:

   RANDOM:

   Nodo 1:
   - Solicitudes: 2
   - Tiempo promedio: 0.3199s

   Nodo 2:
   - Solicitudes: 3
   - Tiempo promedio: 0.5017s

   Nodo 3:
   - Solicitudes: 5
   - Tiempo promedio: 0.4069s

   ROUND ROBIN:

   Nodo 1:
   - Solicitudes: 4
   - Tiempo promedio: 0.3050s

   Nodo 2:
   - Solicitudes: 3
   - Tiempo promedio: 0.5015s

   Nodo 3:
   - Solicitudes: 3
   - Tiempo promedio: 0.4024s

FIN DEL ANÁLISIS COMPARATIVO


---

## 🚀 Parte 6: Test con 100 Solicitudes Concurrentes

Ahora escalamos la prueba a **100 solicitudes concurrentes** para evaluar el rendimiento del sistema.

In [8]:
# ============================================================================
# Prueba de escalabilidad: 100 solicitudes concurrentes
# ============================================================================

# Reinicializar nodos para prueba de escalabilidad
nodes_scale = [
    SimulatedNode(node_id=1, port=8081, processing_time=0.3),
    SimulatedNode(node_id=2, port=8082, processing_time=0.5),
    SimulatedNode(node_id=3, port=8083, processing_time=0.4)
]

async def test_scalability():
    """
    Prueba de escalabilidad con 100 solicitudes concurrentes.
    
    Evalúa cómo el sistema maneja una carga significativa
    de solicitudes simultáneas.
    """
    
    print("\n" + "="*80)
    print("PRUEBA DE ESCALABILIDAD: 100 Solicitudes Concurrentes")
    print("="*80 + "\n")
    
    # Crear balanceador con estrategia round robin
    balancer = LoadBalancer(nodes=nodes_scale, strategy='round_robin')
    
    # Número de solicitudes
    num_requests = 100
    print(f"📨 Enviando {num_requests} solicitudes de forma concurrente...\n")
    
    # Registrar tiempo de inicio
    start_time = time.time()
    
    # Crear todas las solicitudes
    tasks = []
    for i in range(num_requests):
        request_data = {
            'request_id': f'scale_req_{i+1:04d}',
            'image': np.random.rand(224, 224, 3),
            'timestamp': datetime.now().isoformat()
        }
        task = balancer.forward_request(request_data)
        tasks.append(task)
    
    # Ejecutar todas las solicitudes concurrentemente
    results = await asyncio.gather(*tasks)
    
    # Calcular tiempo total
    total_time = time.time() - start_time
    
    # Contar solicitudes exitosas
    successful_requests = len([r for r in results if 'error' not in r])
    
    # Obtener estadísticas
    stats = balancer.get_statistics()
    
    # Mostrar resultados
    print(f"\n✅ RESULTADOS DE LA PRUEBA DE ESCALABILIDAD:\n")
    print(f"   Solicitudes totales: {num_requests}")
    print(f"   Solicitudes exitosas: {successful_requests}")
    print(f"   Tiempo total: {total_time:.2f}s")
    print(f"   Tiempo promedio por solicitud: {total_time/num_requests:.4f}s")
    print(f"   Solicitudes por segundo: {num_requests/total_time:.2f} req/s\n")
    
    print(f"   Distribución de carga:")
    for node_id in sorted(stats['request_distribution'].keys()):
        count = stats['request_distribution'][node_id]
        percentage = (count / num_requests) * 100
        print(f"   - Nodo {node_id}: {count} solicitudes ({percentage:.1f}%)")
    
    print(f"\n   Rendimiento por nodo:")
    for node_stat in stats['nodes_statistics']:
        print(f"\n   Nodo {node_stat['node_id']}:")
        print(f"   - Solicitudes procesadas: {node_stat['total_requests']}")
        print(f"   - Tiempo promedio: {node_stat['avg_processing_time']:.4f}s")
        print(f"   - Tiempo total: {node_stat['total_processing_time']:.2f}s")
    
    return stats, total_time, successful_requests

# Ejecutar la prueba
stats_scale, total_time, successful = await test_scalability()


PRUEBA DE ESCALABILIDAD: 100 Solicitudes Concurrentes

📨 Enviando 100 solicitudes de forma concurrente...


✅ RESULTADOS DE LA PRUEBA DE ESCALABILIDAD:

   Solicitudes totales: 100
   Solicitudes exitosas: 100
   Tiempo total: 1.03s
   Tiempo promedio por solicitud: 0.0103s
   Solicitudes por segundo: 97.46 req/s

   Distribución de carga:
   - Nodo 1: 34 solicitudes (34.0%)
   - Nodo 2: 33 solicitudes (33.0%)
   - Nodo 3: 33 solicitudes (33.0%)

   Rendimiento por nodo:

   Nodo 1:
   - Solicitudes procesadas: 34
   - Tiempo promedio: 0.3094s
   - Tiempo total: 10.52s

   Nodo 2:
   - Solicitudes procesadas: 33
   - Tiempo promedio: 0.5114s
   - Tiempo total: 16.88s

   Nodo 3:
   - Solicitudes procesadas: 33
   - Tiempo promedio: 0.4079s
   - Tiempo total: 13.46s


---

## 🎓 Parte 7: Conclusiones y Recomendaciones

Análisis final de los resultados obtenidos.

In [9]:
# ============================================================================
# Conclusiones y recomendaciones
# ============================================================================

print("\n" + "="*80)
print("CONCLUSIONES Y RECOMENDACIONES")
print("="*80 + "\n")

print("📌 HALLAZGOS PRINCIPALES:\n")

print("1. ESTRATEGIA RANDOM:")
print("   ✓ Ventajas:")
print("     - Implementación simple")
print("     - No requiere estado")
print("     - Buena para distribuciones uniformes")
print("   ✗ Desventajas:")
print("     - Distribución puede ser desigual con pocas solicitudes")
print("     - No considera estado actual del nodo\n")

print("2. ESTRATEGIA ROUND ROBIN:")
print("   ✓ Ventajas:")
print("     - Distribución perfectamente uniforme")
print("     - Predecible y determinista")
print("     - Bueno cuando nodos tienen capacidades similares")
print("   ✗ Desventajas:")
print("     - No adapta a nodos lentos/rápidos")
print("     - Requiere mantener estado\n")

print("3. ESCALABILIDAD:")
print(f"   - Sistema procesó {successful} de 100 solicitudes")
print(f"   - Tiempo total: {total_time:.2f} segundos")
print(f"   - Throughput: {successful/total_time:.2f} solicitudes/segundo")
print(f"   - Latencia promedio: {total_time/successful*1000:.2f} ms\n")

print("📋 RECOMENDACIONES PARA PRODUCCIÓN:\n")

print("1. PARA SISTEMAS CON NODOS HOMOGÉNEOS:")
print("   → Usar Round Robin")
print("   → Garantiza carga uniforme\n")

print("2. PARA SISTEMAS CON NODOS HETEROGÉNEOS:")
print("   → Usar Least Connections o Weighted Round Robin")
print("   → Adapta a capacidades diferentes\n")

print("3. PARA MÁXIMA DISPONIBILIDAD:")
print("   → Implementar health checks")
print("   → Excluir nodos no saludables del pool")
print("   → Usar reintentos automáticos\n")

print("4. PARA MONITOREO:")
print("   → Recopilar métricas de latencia")
print("   → Monitorear distribución de carga")
print("   → Alertas sobre desequilibrio\n")

print("5. TECNOLOGÍAS RECOMENDADAS:")
print("   → NGINX: Balanceo de carga en producción")
print("   → HAProxy: Balance avanzado y monitoreo")
print("   → Kubernetes: Orquestación y balanceo automático")
print("   → gRPC: Comunicación eficiente entre nodos")

print("\n" + "="*80)
print("✅ LABORATORIO COMPLETADO EXITOSAMENTE")
print("="*80)


CONCLUSIONES Y RECOMENDACIONES

📌 HALLAZGOS PRINCIPALES:

1. ESTRATEGIA RANDOM:
   ✓ Ventajas:
     - Implementación simple
     - No requiere estado
     - Buena para distribuciones uniformes
   ✗ Desventajas:
     - Distribución puede ser desigual con pocas solicitudes
     - No considera estado actual del nodo

2. ESTRATEGIA ROUND ROBIN:
   ✓ Ventajas:
     - Distribución perfectamente uniforme
     - Predecible y determinista
     - Bueno cuando nodos tienen capacidades similares
   ✗ Desventajas:
     - No adapta a nodos lentos/rápidos
     - Requiere mantener estado

3. ESCALABILIDAD:
   - Sistema procesó 100 de 100 solicitudes
   - Tiempo total: 1.03 segundos
   - Throughput: 97.46 solicitudes/segundo
   - Latencia promedio: 10.26 ms

📋 RECOMENDACIONES PARA PRODUCCIÓN:

1. PARA SISTEMAS CON NODOS HOMOGÉNEOS:
   → Usar Round Robin
   → Garantiza carga uniforme

2. PARA SISTEMAS CON NODOS HETEROGÉNEOS:
   → Usar Least Connections o Weighted Round Robin
   → Adapta a capacidades d

---

## 🔗 Recursos Adicionales

### Documentación Oficial
- [Python asyncio](https://docs.python.org/3/library/asyncio.html)
- [aiohttp Documentation](https://docs.aiohttp.org/)
- [NGINX Load Balancing](https://nginx.org/en/docs/http/load_balancing.html)
- [HAProxy Configuration](http://www.haproxy.org/)

### Libros Recomendados
- "Designing Data-Intensive Applications" - Martin Kleppmann
- "System Design Interview" - Alex Xu
- "Web Scalability for Startup Engineers" - Artur Ejsmont

### Conceptos para Profundizar
- **Consistent Hashing**: Distribución hash consistente para escalabilidad
- **Circuit Breaker Pattern**: Manejo de fallos en sistemas distribuidos
- **Rate Limiting**: Control de velocidad de solicitudes
- **Caching Strategies**: Implementación de cachés distribuidos
- **Service Mesh**: Comunicación avanzada entre servicios

---

## 📝 Notas Finales

Este laboratorio introduce los conceptos fundamentales de balanceo de carga en sistemas distribuidos. En un entorno de producción, es importante considerar:

- **Tolerancia a fallos**: Qué sucede si un nodo se cae
- **Latencia de red**: Tiempo de comunicación entre balanceador y nodos
- **Escalabilidad horizontal**: Agregar más nodos dinámicamente
- **Monitoreo y observabilidad**: Métricas en tiempo real
- **Seguridad**: Autenticación y encriptación entre componentes

¡Felicidades por completar esta actividad!